# 🔬 Notebook 3: Key-Value Store — Deep Dives


## 🛠️ Setup

```bash
cd 06-system-designs/key-value-store
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1 — consistent hashing (with and without vnodes)

### 😱 Bad: one position per node

When each node has a single ring position, load imbalance gets ugly with small N:
a node that happens to sit just after a long empty arc owns *all* those keys.


In [ ]:
import hashlib, bisect
from collections import defaultdict

def h(s: str) -> int:
    return int(hashlib.md5(s.encode()).hexdigest(), 16)

class Ring:
    def __init__(self, vnodes_per_node: int = 1):
        self.vnodes_per_node = vnodes_per_node
        self.ring: list[tuple[int, str]] = []
        self.positions: list[int] = []

    def add_node(self, node: str):
        for i in range(self.vnodes_per_node):
            self.ring.append((h(f"{node}#{i}"), node))
        self.ring.sort()
        self.positions = [p for p, _ in self.ring]

    def owner(self, key: str) -> str:
        i = bisect.bisect_right(self.positions, h(key)) % len(self.ring)
        return self.ring[i][1]

    def replicas(self, key: str, n: int) -> list[str]:
        i = bisect.bisect_right(self.positions, h(key)) % len(self.ring)
        out, seen = [], set()
        while len(out) < n and len(seen) < len(set(n for _, n in self.ring)):
            node = self.ring[i % len(self.ring)][1]
            if node not in seen:
                seen.add(node); out.append(node)
            i += 1
        return out

def load_dist(ring, n_keys=20_000):
    d = defaultdict(int)
    for k in range(n_keys):
        d[ring.owner(f"k{k}")] += 1
    return d

bad = Ring(vnodes_per_node=1)
for node in ["A", "B", "C", "D", "E"]:
    bad.add_node(node)
dist = load_dist(bad)
print("1 vnode / node  →", dict(dist))
print(f"   spread: min={min(dist.values())}  max={max(dist.values())}  "
      f"ratio={max(dist.values())/min(dist.values()):.2f}x")

### ✅ Best: many vnodes per node smooths the load

In [ ]:
good = Ring(vnodes_per_node=128)
for node in ["A", "B", "C", "D", "E"]:
    good.add_node(node)
dist = load_dist(good)
print("128 vnodes / node →", dict(dist))
print(f"   spread: min={min(dist.values())}  max={max(dist.values())}  "
      f"ratio={max(dist.values())/min(dist.values()):.2f}x")
print("➡ Much more even. Also: removing a node redistributes its share "
      "across ALL remaining nodes, not just one unlucky neighbor.")
# The other half of the ring's job: choosing WHICH N nodes hold each key.
# Walk clockwise from the key's position, skipping vnodes of a node we already picked.
print("\nReplica placement (N=3) — the preference list for a few keys:")
for k in ("user:1", "user:2", "cart:99", "session:abc"):
    print(f"  {k:<12} -> {good.replicas(k, 3)}")

from collections import Counter
lead = Counter(good.replicas(f"k{i}", 3)[0] for i in range(20_000))
print(f"\nPrimary-replica share across 20k keys: {dict(sorted(lead.items()))}")
print("Each node is primary for ~1/5 of keys AND a backup replica for ~2/5 more.")
print("\nThe cost of vnodes, since the section only sold the benefits:")
print("  - The ring metadata is now 128x bigger and must be gossiped to every node.")
print("  - A failed node's data is spread over ALL peers, so recovery reads from all of")
print("    them — faster to rebuild, but a wider blast radius for a correlated failure.")
print("  - Range scans are dead: adjacent keys land on unrelated nodes by construction.")


## Deep dive 2 — a tiny quorum simulator

We simulate N replicas that sometimes drop requests (`fail_prob`). We try different
W/R settings and watch what happens.


In [ ]:
import random

class Replica:
    """A replica that sometimes drops a request (slow node, packet loss, GC pause)."""
    def __init__(self, name, fail_prob=0.0):
        self.name = name
        self.store: dict[str, tuple[str, int]] = {}
        self.fail_prob = fail_prob

    def write(self, key, value, version):
        if random.random() < self.fail_prob:
            return False
        cur = self.store.get(key)
        if cur is None or version > cur[1]:
            self.store[key] = (value, version)
        return True

    def read(self, key):
        if random.random() < self.fail_prob:
            return None
        return self.store.get(key)

class Coordinator:
    def __init__(self, replicas, N, W, R):
        self.replicas = replicas
        self.N, self.W, self.R = N, W, R
        self.version = 0

    def put(self, key, value):
        """Send to ALL N replicas, succeed once W have acked.

        Sending to all N and *waiting* for only W is the real Dynamo behaviour, and it
        matters: stopping the moment W acks arrive would leave the remaining replicas
        permanently behind and quietly break the R+W>N guarantee.
        """
        self.version += 1
        acks = sum(1 for r in self.replicas if r.write(key, value, self.version))
        return {"status": "ok" if acks >= self.W else "fail",
                "acks": acks, "version": self.version}

    def get(self, key):
        """Read from all N, return once R have answered, reconcile by highest version."""
        responses = [v for v in (r.read(key) for r in self.replicas) if v is not None]
        if len(responses) < self.R:
            return {"status": "fail", "reason": f"only {len(responses)}/{self.R} replicas answered"}
        return {"status": "ok", "value": max(responses[:self.R], key=lambda x: x[1])}

random.seed(0)
TRIALS = 2_000

print("Availability: how often does a write succeed when each replica drops 20% of requests?")
for W in (1, 2, 3):
    reps = [Replica(f"r{i}", fail_prob=0.2) for i in range(3)]
    c = Coordinator(reps, N=3, W=W, R=1)
    fails = sum(1 for i in range(TRIALS) if c.put(f"k{i}", "v")["status"] == "fail")
    print(f"  N=3 W={W}: {fails/TRIALS:6.1%} of writes failed")
print("➡ Higher W = stronger guarantee, lower availability. That is CAP in one line.")


### Does `R + W > N` actually buy anything? Measure it.

The rule says: if reads and writes both go to the same N replicas and `R + W > N`, the read set
and the write set **must overlap**, so a successful read sees at least one replica holding the
newest write. Let's check that claim empirically instead of trusting it.

Two separate things can go wrong, and they need two separate experiments:

1. **Flaky replicas** (packet loss, GC pauses) — a write reaches only *some* replicas, so a
   later read can land on one that missed it. This is what produces **stale reads**.
2. **A dead replica** (an AZ is down) — some R/W settings simply cannot be satisfied any more.
   This is what produces **unavailability**.

In [ ]:
def stale_rate(N, W, R, fail_prob=0.4, trials=4_000, seed=1):
    """Experiment 1: flaky replicas. Write NEW, immediately read back, count stale reads."""
    random.seed(seed)
    reps = [Replica(f"r{i}", fail_prob=fail_prob) for i in range(N)]
    c = Coordinator(reps, N=N, W=W, R=R)
    for r in reps:
        r.store["k"] = ("OLD", 0)          # so a stale read is possible at all

    stale = ok = 0
    for _ in range(trials):
        if c.put("k", "NEW")["status"] == "fail":
            continue                        # write rejected; nothing to check
        got = c.get("k")
        if got["status"] == "fail":
            continue                        # read rejected; not a staleness event
        ok += 1
        if got["value"][0] != "NEW":
            stale += 1
        for r in reps:                      # reset for an independent trial
            r.store["k"] = ("OLD", 0)
        c.version = 0
    return stale / max(ok, 1)

def unavail_rate(N, W, R, dead=1, trials=2_000, seed=1):
    """Experiment 2: `dead` replicas are permanently gone. How much traffic can we serve?"""
    random.seed(seed)
    reps = [Replica(f"r{i}", fail_prob=1.0 if i < dead else 0.0) for i in range(N)]
    c = Coordinator(reps, N=N, W=W, R=R)
    failed = 0
    for i in range(trials):
        if c.put("k", f"v{i}")["status"] == "fail" or c.get("k")["status"] == "fail":
            failed += 1
    return failed / trials

print(f"N=3.  Stale reads: 40% flaky replicas.  Unavailable: 1 of 3 replicas dead.")
print(f"  {'setting':<26}{'stale reads':>13}{'unavailable':>14}")
for W, R in [(1, 1), (2, 2), (1, 3), (3, 1)]:
    rule = "R+W>N ✓" if R + W > 3 else "R+W≤N ✗"
    label = f"W={W} R={R}  ({rule})"
    print(f"  {label:<26}{stale_rate(3, W, R):>12.1%}{unavail_rate(3, W, R):>14.1%}")

print("""
Read the table as a menu, not a ranking — every row pays for its guarantee somewhere:

  W=1 R=1  R+W ≤ N, and the stale-read column proves it is not a theoretical concern.
           Fast and almost always up. Returning yesterday's value is the deal, not a bug.
  W=2 R=2  zero stale reads AND survives one dead replica. This is why plain QUORUM is
           the default that everyone recommends.
  W=1 R=3  zero stale reads and cheap writes — but reads need EVERY replica, so a single
           dead node makes 100% of reads fail.
  W=3 R=1  the mirror image: reads are cheap, one dead node kills every write.

So R+W>N is about CORRECTNESS and says nothing about AVAILABILITY. For that you separately
need W ≤ live_replicas and R ≤ live_replicas. Both W=1/R=3 and W=3/R=1 satisfy the
arithmetic and are still useless the moment one machine reboots.""")


## Deep dive 3 — anti-entropy with Merkle trees

Two replicas **drift** apart (a node was down; writes went to its peers). When it
returns, we want to find the **missing keys** without comparing the entire dataset.

A Merkle tree builds a hash tree over the keyspace. Two replicas compare their
roots; if roots match, they're in sync — **zero** key comparisons. If roots
differ, we recurse **only into differing subtrees**. For small diffs, bandwidth
is `O(log N)` instead of `O(N)`.

```
        H(root)
        /     \
     H(L)     H(R)      ← compare roots; differ → recurse
     /  \     /  \
   ...  ...  ...  ...    ← only descend into differing subtrees
```

Cassandra uses this for **repair**, DynamoDB and Riak for **hinted handoff / read repair**.
Let's build one:


In [ ]:
import hashlib
from dataclasses import dataclass, field

def H(b: bytes) -> str:
    return hashlib.sha1(b).hexdigest()[:10]   # short hashes for readable output

@dataclass
class MerkleNode:
    hash: str
    left: "MerkleNode | None" = None
    right: "MerkleNode | None" = None
    keys: list[str] = field(default_factory=list)   # only on leaves

def build_merkle(items: dict[str, str], bucket_size: int = 4) -> MerkleNode:
    """Bucket keys by sorted order, hash each bucket as a leaf, then fold up."""
    sorted_items = sorted(items.items())
    # Leaves
    leaves = []
    for i in range(0, len(sorted_items), bucket_size):
        chunk = sorted_items[i:i+bucket_size]
        payload = "|".join(f"{k}={v}" for k, v in chunk).encode()
        leaves.append(MerkleNode(H(payload), keys=[k for k, _ in chunk]))
    if not leaves:
        return MerkleNode(H(b""))
    # Fold pairwise until one root
    while len(leaves) > 1:
        nxt = []
        for i in range(0, len(leaves), 2):
            l = leaves[i]
            r = leaves[i+1] if i+1 < len(leaves) else leaves[i]
            nxt.append(MerkleNode(H((l.hash + r.hash).encode()), left=l, right=r))
        leaves = nxt
    return leaves[0]

def diff(a: MerkleNode, b: MerkleNode, out: list[str]) -> int:
    """Return number of leaf comparisons performed; collect differing keys."""
    if a.hash == b.hash:
        return 1                                  # 1 comparison, done
    if a.left is None and b.left is None:         # both leaves
        out.extend(set(a.keys) | set(b.keys))
        return 1
    return 1 + diff(a.left, b.left, out) + diff(a.right, b.right, out)

# Replica A and B differ on just 2 keys out of 32
A = {f"k{i:02d}": f"v{i}"          for i in range(32)}
B = dict(A)
B["k05"] = "v5-CHANGED"
B["k20"] = "v20-CHANGED"

tA, tB = build_merkle(A), build_merkle(B)
differing, comparisons = [], 0
comparisons = diff(tA, tB, differing)
print(f"Naive comparison:  would need 32 key checks.")
print(f"Merkle tree:       {comparisons} node-hash comparisons.")
print(f"Detected drift on: {sorted(set(differing))}")

## Deep dive 4 — hinted handoff + read repair

What if a replica is temporarily down when we try to write? We don't fail — we
write to a **substitute** node with a "hint" saying *"this really belongs to N3;
deliver it when N3 comes back"*. When N3 returns, the hint is replayed.

**Read repair** is the other side: on every read, if replicas return different
versions, the coordinator pushes the newest version back to the stale ones.

Tiny simulation:


In [ ]:
class NodeWithHints:
    def __init__(self, name):
        self.name = name
        self.store: dict[str, tuple[str, int]] = {}
        self.hints: list[tuple[str, str, str, int]] = []   # (target, k, v, ver)
        self.alive = True

    def write(self, k, v, ver):
        if not self.alive:
            return False
        cur = self.store.get(k)
        if cur is None or ver > cur[1]:
            self.store[k] = (v, ver)
        return True

    def store_hint(self, target, k, v, ver):
        self.hints.append((target, k, v, ver))

    def flush_hints(self, cluster):
        delivered = 0
        remaining = []
        for target, k, v, ver in self.hints:
            if cluster[target].write(k, v, ver):
                delivered += 1
            else:
                remaining.append((target, k, v, ver))
        self.hints = remaining
        return delivered

nodes = {n: NodeWithHints(n) for n in ["A", "B", "C"]}
nodes["C"].alive = False                   # C is down

# Client writes key "x" — replicas are [A, B, C]. C is down → A takes a hint for C.
ver = 1
nodes["A"].write("x", "hello", ver)
nodes["B"].write("x", "hello", ver)
nodes["A"].store_hint("C", "x", "hello", ver)
print("C store before recovery:", nodes["C"].store)
print("A holds hints:          ", nodes["A"].hints)

# C comes back — A replays hints to it
nodes["C"].alive = True
delivered = nodes["A"].flush_hints(nodes)
print(f"After recovery: {delivered} hint(s) delivered")
print("C store after recovery: ", nodes["C"].store)

## Deep dive 5 — putting it all together

A production read path in this design looks like:

1. Client hits any node → that node is the **coordinator**.
2. Coordinator hashes the key, finds the **N replicas** on the ring.
3. It sends the read to all N, waits for **R** responses.
4. If responses disagree, it **reconciles** (pick highest version / return siblings).
5. It **read-repairs** stale replicas in the background.
6. Anti-entropy (Merkle trees) runs periodically to catch anything reads missed.

Write path is symmetric: hash → N replicas → wait for W acks → store hints for
any replica that didn't answer.

That's the whole Dynamo paper in one page. 🎉

### Where to go next
- [Amazon's Dynamo paper (2007)](https://www.allthingsdistributed.com/files/amazon-dynamo-sosp2007.pdf)
- Cassandra docs on [consistent hashing](https://cassandra.apache.org/doc/stable/cassandra/architecture/dynamo.html)
- Designing Data-Intensive Applications, Chapter 5–6 (replication & partitioning)
